# Music Annotation Project
## Notebook 77A: Exact Dark-Collar Rescoring of the Locked Notebook 77 Paper Veto

**Runtime:** CPU. Select **Runtime → Run all**.

Notebook 77A repairs the scoring layer of Notebook 77 without changing its locked observable decisions.

### Required sequence

1. Identify one completed Notebook 77 run and its source Notebook 75 / 75A runs.
2. Verify the Notebook 77 prediction lock and all available page/output hashes.
3. Load the already locked `PAPER_SUPPORTED` masks and A2 pages. Do not rebuild or retune the paper-veto rule.
4. Recreate the exact Notebook 75 dark-collar scoring population and truth labels.
5. **Fail closed unless rescored C1 reproduces Notebook 75 / 75A exactly**, pooled and page by page.
6. Only after exact reproduction, apply the same scorer to locked A2.
7. Determine whether A2 removes retained annotation without sacrificing genuine print recovery.

### Scoring population

Notebook 75 evaluates restored **observed dark collar material**, not every byte difference between C0 and C1:

```text
selected arm pixel
    = arm differs from C0
    AND arm byte equals observed byte
    AND observed intensity < 200
    AND pixel belongs to the arm's collar population
```

Truth is then assigned using the same frozen clean printed-ink map and one-pixel registration-tolerance / truth-uncertain construction used in the project lineage.

### Non-negotiable exactness gate

Notebook 77A does not accept a merely similar C1 total. It compares page-level:

- true print recovered;
- annotation retained;
- truth-uncertain retained;
- utility;
- catastrophic components;
- selected/restored dark pixels when present in the Notebook 75 reference.

Any mismatch stops interpretation and writes a mismatch table. A2 is not judged from an unproven scorer.

### Authority

Notebook 77A is a read-only scoring repair and audit. It does not change Notebook 77, its predeclared policy, its prediction lock, C0, C1, A2, `v8_8_operator.py`, or production outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib seaborn

## 1. Configuration and source-run resolution

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, zipfile, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB77_ROOT=ROOT/'notebook77_observable_paper_veto'
NB75_ROOT=ROOT/'notebook75_v88_collar_preservation_counterfactual'
NB75A_ROOT=ROOT/'notebook75a_stratified_reanalysis'
OUTPUT_ROOT=ROOT/'notebook77a_exact_dark_collar_rescore'

SOURCE_NB77_RUN_OVERRIDE=None
SOURCE_NB75_RUN_OVERRIDE=None
SOURCE_NB75A_RUN_OVERRIDE=None

DARK_THRESHOLD=200
BOOTSTRAP_RESAMPLES=10000
SEED=42
EXPECTED_C1_TRUE_PRINT=18178
EXPECTED_C1_ANNOTATION=1246
EXPECTED_C1_CATASTROPHES=1
EXPECTED_C1_UTILITY=11448

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook77a_exact_dark_collar_rescore'
OUT=OUTPUT_ROOT/RUN; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; VISUAL=OUT/'visuals'; REVIEW=OUT/'review'; UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,VISUAL,REVIEW,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(x,msg):
    if not bool(x):raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp');t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str));t.replace(path)
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

NB77=Path(SOURCE_NB77_RUN_OVERRIDE) if SOURCE_NB77_RUN_OVERRIDE else latest(NB77_ROOT,['prediction_lock.json','audit/page_output_hashes.csv','observable_maps','pages','tables/results_by_page.csv'])
require(NB77 is not None,'No complete Notebook 77 run found.')
lock77=json.loads((NB77/'prediction_lock.json').read_text())
source_record=json.loads((NB77/'audit'/'source_record.json').read_text()) if (NB77/'audit'/'source_record.json').exists() else {}
NB75=Path(SOURCE_NB75_RUN_OVERRIDE) if SOURCE_NB75_RUN_OVERRIDE else Path(source_record.get('notebook75','')) if source_record.get('notebook75') else latest(NB75_ROOT,['tables/results_by_page.csv','pages','prediction_lock/prediction_lock.json'])
NB75A=Path(SOURCE_NB75A_RUN_OVERRIDE) if SOURCE_NB75A_RUN_OVERRIDE else Path(source_record.get('notebook75a','')) if source_record.get('notebook75a') else latest(NB75A_ROOT,['tables/collar_exposed_arm_summary.csv','tables/page_role_resolution.csv'])
require(NB75 is not None and NB75.exists(),'Notebook 75 source run not found.')
require(NB75A is not None and NB75A.exists(),'Notebook 75A source run not found.')
print('Notebook 77:',NB77);print('Notebook 75:',NB75);print('Notebook 75A:',NB75A);print('Notebook 77A:',OUT)

## 2. Verify locked Notebook 77 identity and immutable A2 decisions

In [ ]:
require(lock77.get('state')=='OBSERVABLE_DECISIONS_LOCKED_BEFORE_TRUTH',f'Unexpected Notebook 77 lock state: {lock77.get("state")}')
hashes=pd.read_csv(NB77/'audit'/'page_output_hashes.csv')
required_hash_cols={'page_key','observed_array_sha256','C0_array_sha256','C1_array_sha256','A2_array_sha256','paper_array_sha256'}
require(required_hash_cols.issubset(hashes.columns),f'Notebook 77 page hashes missing: {sorted(required_hash_cols-set(hashes.columns))}')

verify=[]
for _,r in hashes.iterrows():
    key=str(r.page_key);d=NB77/'pages'/key;m=NB77/'observable_maps'/f'{key}.npz'
    paths={'observed':d/'observed.png','C0':d/'C0_INCUMBENT.png','C1':d/'C1_RING2.png','A2':d/'A2_PAPER_VETO.png'}
    for name,p in paths.items():require(p.exists(),f'{key}: missing locked {name} image')
    ims={name:cv2.imread(str(p),0) for name,p in paths.items()}
    require(all(v is not None for v in ims.values()),f'{key}: unreadable locked image')
    require(m.exists(),f'{key}: locked observable map missing')
    with np.load(m) as z:
        require('paper_supported' in z.files,f'{key}: paper_supported absent')
        paper=z['paper_supported']>0
    checks={
      'observed':array_hash(ims['observed'])==str(r.observed_array_sha256),
      'C0':array_hash(ims['C0'])==str(r.C0_array_sha256),
      'C1':array_hash(ims['C1'])==str(r.C1_array_sha256),
      'A2':array_hash(ims['A2'])==str(r.A2_array_sha256),
      'paper':array_hash(paper.astype(np.uint8))==str(r.paper_array_sha256),
      'A2_non_veto_equals_C1':bool(np.array_equal(ims['A2'][~paper],ims['C1'][~paper])),
      'A2_veto_equals_C0':bool(np.array_equal(ims['A2'][paper],ims['C0'][paper])),
    }
    verify.append({'page_key':key,**checks})
verification=pd.DataFrame(verify);verification.to_csv(AUDIT/'notebook77_lock_replay.csv',index=False)
require(verification.drop(columns='page_key').all().all(),'Notebook 77 lock replay failed. See audit/notebook77_lock_replay.csv')
print('Notebook 77 lock replay passed for',len(verification),'pages.')

## 3. Load the authoritative Notebook 75 page-level reference

In [ ]:
ref75=pd.read_csv(NB75/'tables'/'results_by_page.csv')
ref75=ref75[(ref75.population=='PAIRED_DEVELOPMENT')&ref75.arm.isin(['C1_PRESERVE_RING2','C2_PRESERVE_RING1_RING2'])].copy()
require(len(ref75)>0,'Notebook 75 paired reference is empty.')
roles=pd.read_csv(NB75A/'tables'/'page_role_resolution.csv')
exposed_keys=set(roles.loc[roles.c2_restored_pixels>0,'page_key'].astype(str))
ref75=ref75[ref75.page_key.astype(str).isin(exposed_keys)].copy()
require(ref75.page_key.nunique()==len(verification),'Notebook 75 exposed roster and Notebook 77 lock roster differ.')

c1_ref=ref75[ref75.arm=='C1_PRESERVE_RING2'].copy()
required_ref={'page_key','page','source_group','true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components'}
require(required_ref.issubset(c1_ref.columns),f'Notebook 75 reference missing: {sorted(required_ref-set(c1_ref.columns))}')
pooled_reference={
 'true_print_recovered':int(c1_ref.true_print_recovered.sum()),
 'annotation_retained':int(c1_ref.annotation_retained.sum()),
 'truth_uncertain_retained':int(c1_ref.truth_uncertain_retained.sum()),
 'utility':int(c1_ref.utility.sum()),
 'catastrophic_components':int(c1_ref.catastrophic_components.sum()),
}
require(pooled_reference['true_print_recovered']==EXPECTED_C1_TRUE_PRINT,'Notebook 75 C1 true-print total differs from declared reference.')
require(pooled_reference['annotation_retained']==EXPECTED_C1_ANNOTATION,'Notebook 75 C1 annotation total differs from declared reference.')
require(pooled_reference['utility']==EXPECTED_C1_UTILITY,'Notebook 75 C1 utility differs from declared reference.')
require(pooled_reference['catastrophic_components']==EXPECTED_C1_CATASTROPHES,'Notebook 75 C1 catastrophe total differs from declared reference.')
atomic_json(AUDIT/'notebook75_c1_reference.json',pooled_reference)
print(json.dumps(pooled_reference,indent=2))

## 4. Exact Notebook 75 dark-collar scorer

This scorer is deliberately narrow. It scores only dark observed bytes restored relative to C0 and uses the established adaptive clean-ink map plus one-pixel tolerance/boundary exclusion. The next cell must reproduce every C1 page row before A2 is interpreted.

In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
clean_map=image_map(ROOT/'source'/'clean')

def printed_ink(clean):return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def truth_maps(clean):
    ink=printed_ink(clean);k=np.ones((3,3),np.uint8);near=cv2.dilate(ink.astype(np.uint8),k)>0;eroded=cv2.erode(ink.astype(np.uint8),k)>0
    uncertain=np.logical_xor(near,eroded)
    # Notebook 75 PRINT uses registered clean ink within one-pixel tolerance, excluding unstable boundary.
    print_truth=near&~uncertain
    annotation_truth=~near&~uncertain
    return print_truth,annotation_truth,uncertain

def component_catastrophes(selected,tp,fp,mask):
    n,labels,_,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);cats=0
    for cid in sorted(set(labels[selected].tolist())-{0}):
        cm=labels==cid;ct=int((tp&cm).sum());cf=int((fp&cm).sum())
        if cf>=50 or (cf/max(ct,1)>1.0 and cf>=20):cats+=1
    return cats

def score_arm(observed,c0,candidate,mask,print_truth,annotation_truth,truth_uncertain):
    selected=(candidate!=c0)&(candidate==observed)&(observed<DARK_THRESHOLD)&mask
    tp=selected&print_truth;fp=selected&annotation_truth;tu=selected&truth_uncertain
    cats=component_catastrophes(selected,tp,fp,mask)
    return {'selected_dark_pixels':int(selected.sum()),'true_print_recovered':int(tp.sum()),'annotation_retained':int(fp.sum()),'truth_uncertain_retained':int(tu.sum()),'catastrophic_components':cats,'utility':int(tp.sum()-5*fp.sum()-500*cats)},selected,tp,fp,tu

## 5. Prove exact C1 reproduction before calculating A2

In [ ]:
# Authoritative prepared masks come from the Notebook 75/V7.5 lineage referenced by Notebook 77.
v75_reference=Path(source_record.get('v75_cache','')) if source_record.get('v75_cache') else None
if v75_reference is None or not v75_reference.exists():
    # Notebook 77 source record may use v75_cache as a run path.
    v75_reference=latest(ROOT/'reconstruction_geometry_benchmark_v7_5',['tables/cache_status.csv'])
require(v75_reference is not None,'No V7.5 cache run found for authoritative masks.')
status=pd.read_csv(v75_reference/'tables'/'cache_status.csv');cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}

reproduced=[];runtime={}
ref_lookup=c1_ref.set_index('page_key')
for _,hr in hashes.iterrows():
    key=str(hr.page_key);ref=ref_lookup.loc[key];page=str(ref.page);d=NB77/'pages'/key
    observed=cv2.imread(str(d/'observed.png'),0);c0=cv2.imread(str(d/'C0_INCUMBENT.png'),0);c1=cv2.imread(str(d/'C1_RING2.png'),0);a2=cv2.imread(str(d/'A2_PAPER_VETO.png'),0)
    require(page in clean_map,f'{page}: clean truth missing');require(page in cache,f'{page}: authoritative cache missing')
    clean=cv2.imread(str(clean_map[page]),0)
    with np.load(cache[page]) as z: mask=z['mask']>0
    require(clean.shape==observed.shape==mask.shape,f'{key}: shape mismatch')
    pt,at,ut=truth_maps(clean)
    calc,selected,tp,fp,tu=score_arm(observed,c0,c1,mask,pt,at,ut)
    row={'page_key':key,'page':page,'source_group':str(ref.source_group),**calc}
    for field in ['true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components']:
        row['reference_'+field]=int(ref[field]);row['delta_'+field]=int(calc[field]-int(ref[field]))
    if 'restored_pixels' in ref.index:
        row['reference_restored_pixels']=int(ref.restored_pixels)
    reproduced.append(row)
    runtime[key]={'page':page,'source_group':str(ref.source_group),'observed':observed,'c0':c0,'c1':c1,'a2':a2,'mask':mask,'pt':pt,'at':at,'ut':ut,'c1_selected':selected,'c1_tp':tp,'c1_fp':fp,'c1_tu':tu}

reproduction=pd.DataFrame(reproduced);reproduction.to_csv(AUDIT/'c1_exact_reproduction_by_page.csv',index=False)
delta_cols=[c for c in reproduction.columns if c.startswith('delta_')]
exact_pagewise=bool((reproduction[delta_cols]==0).all().all())
pooled_calc={k:int(reproduction[k].sum()) for k in ['true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components']}
pooled_exact=all(pooled_calc[k]==pooled_reference[k] for k in pooled_reference)
repro_gate={'pagewise_exact':exact_pagewise,'pooled_exact':pooled_exact,'calculated':pooled_calc,'reference':pooled_reference,'dark_threshold':DARK_THRESHOLD,'pages':len(reproduction)}
atomic_json(AUDIT/'c1_reproduction_gate.json',repro_gate)
if not (exact_pagewise and pooled_exact):
    reproduction.loc[(reproduction[delta_cols]!=0).any(axis=1)].to_csv(AUDIT/'c1_reproduction_mismatches.csv',index=False)
    raise RuntimeError('C1 exact reproduction FAILED. A2 will not be interpreted. See audit/c1_reproduction_mismatches.csv')
print('EXACT C1 REPRODUCTION PASSED');print(json.dumps(repro_gate,indent=2))

## 6. Rescore the already locked A2 decisions with the proven scorer

In [ ]:
a2_rows=[];removed_rows=[]
for key,z in runtime.items():
    calc,sel,tp,fp,tu=score_arm(z['observed'],z['c0'],z['a2'],z['mask'],z['pt'],z['at'],z['ut'])
    c1r=reproduction.set_index('page_key').loc[key]
    a2_rows.append({'page_key':key,'page':z['page'],'source_group':z['source_group'],'arm':'A2_PAPER_VETO',**calc})
    with np.load(NB77/'observable_maps'/f'{key}.npz') as q: paper=q['paper_supported']>0
    removed=z['c1_selected']&~sel
    removed_rows.append({'page_key':key,'page':z['page'],'source_group':z['source_group'],'paper_supported_pixels':int(paper.sum()),'c1_dark_selected_removed':int(removed.sum()),'true_print_removed':int((removed&z['pt']).sum()),'annotation_removed':int((removed&z['at']).sum()),'truth_uncertain_removed':int((removed&z['ut']).sum()),'paper_veto_not_in_dark_population':int((paper&~z['c1_selected']).sum())})

a2_page=pd.DataFrame(a2_rows);removed=pd.DataFrame(removed_rows)
c1_page=reproduction[['page_key','page','source_group','selected_dark_pixels','true_print_recovered','annotation_retained','truth_uncertain_retained','catastrophic_components','utility']].copy();c1_page['arm']='C1_RING2'
results=pd.concat([c1_page,a2_page],ignore_index=True);results.to_csv(TABLES/'rescored_results_by_page.csv',index=False);removed.to_csv(TABLES/'paper_veto_removed_dark_population.csv',index=False)
print(results.groupby('arm')[['selected_dark_pixels','true_print_recovered','annotation_retained','truth_uncertain_retained','utility','catastrophic_components']].sum())
print('\nRemoved by A2 paper veto:');print(removed[['c1_dark_selected_removed','true_print_removed','annotation_removed','truth_uncertain_removed','paper_veto_not_in_dark_population']].sum())

## 7. Pooled summaries and page-clustered uncertainty

In [ ]:
def bootstrap(g):
    rng=np.random.default_rng(SEED);pages=g.page_key.to_numpy();idx=g.set_index('page_key');out=[]
    for _ in range(BOOTSTRAP_RESAMPLES):
        q=idx.loc[rng.choice(pages,size=len(pages),replace=True)];tp=float(q.true_print_recovered.sum());fp=float(q.annotation_retained.sum());cat=float(q.catastrophic_components.sum());out.append((tp-5*fp-500*cat,fp/max(tp,1),fp/max(tp+fp,1)))
    return np.asarray(out,float)
rows=[]
for arm,g in results.groupby('arm'):
    tp=int(g.true_print_recovered.sum());fp=int(g.annotation_retained.sum());tu=int(g.truth_uncertain_retained.sum());cat=int(g.catastrophic_components.sum());b=bootstrap(g)
    rows.append({'arm':arm,'pages':g.page_key.nunique(),'source_groups':g.source_group.nunique(),'selected_dark_pixels':int(g.selected_dark_pixels.sum()),'true_print_recovered':tp,'annotation_retained':fp,'truth_uncertain_retained':tu,'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(tp+fp,1),'utility':tp-5*fp-500*cat,'catastrophic_components':cat,'benefiting_pages_ge10':int((g.true_print_recovered>=10).sum()),'benefiting_source_groups':int(g.loc[g.true_print_recovered>=10,'source_group'].nunique()),'bootstrap_utility_95_low':float(np.quantile(b[:,0],.025)),'bootstrap_utility_95_high':float(np.quantile(b[:,0],.975)),'bootstrap_false_true_95_high':float(np.quantile(b[:,1],.975)),'bootstrap_false_preservation_95_high':float(np.quantile(b[:,2],.975))})
summary=pd.DataFrame(rows);summary.to_csv(TABLES/'rescored_pooled_arm_summary.csv',index=False)
print(summary.to_string(index=False))

## 8. Direct A2 versus C1 decision

In [ ]:
s=summary.set_index('arm');c1=s.loc['C1_RING2'];a2=s.loc['A2_PAPER_VETO']
comparison={
 'c1_true_print':int(c1.true_print_recovered),'a2_true_print':int(a2.true_print_recovered),'true_print_delta':int(a2.true_print_recovered-c1.true_print_recovered),'true_print_retention':float(a2.true_print_recovered/max(c1.true_print_recovered,1)),
 'c1_annotation':int(c1.annotation_retained),'a2_annotation':int(a2.annotation_retained),'annotation_delta':int(a2.annotation_retained-c1.annotation_retained),'annotation_removed':int(c1.annotation_retained-a2.annotation_retained),'annotation_retention':float(a2.annotation_retained/max(c1.annotation_retained,1)),
 'c1_truth_uncertain':int(c1.truth_uncertain_retained),'a2_truth_uncertain':int(a2.truth_uncertain_retained),'truth_uncertain_delta':int(a2.truth_uncertain_retained-c1.truth_uncertain_retained),
 'c1_utility':int(c1.utility),'a2_utility':int(a2.utility),'utility_delta':int(a2.utility-c1.utility),
 'c1_catastrophes':int(c1.catastrophic_components),'a2_catastrophes':int(a2.catastrophic_components),'catastrophe_delta':int(a2.catastrophic_components-c1.catastrophic_components),
 'pages_with_true_print_loss':int((results.pivot(index='page_key',columns='arm',values='true_print_recovered')['A2_PAPER_VETO']<results.pivot(index='page_key',columns='arm',values='true_print_recovered')['C1_RING2']).sum()),
 'pages_with_annotation_reduction':int((results.pivot(index='page_key',columns='arm',values='annotation_retained')['A2_PAPER_VETO']<results.pivot(index='page_key',columns='arm',values='annotation_retained')['C1_RING2']).sum()),
}
comparison['improves_without_any_genuine_print_loss']=bool(comparison['true_print_delta']==0 and comparison['annotation_delta']<0 and comparison['pages_with_true_print_loss']==0)
atomic_json(TABLES/'A2_vs_C1_exact_comparison.json',comparison)
print(json.dumps(comparison,indent=2))

## 9. Source-group and annotator/style stability

In [ ]:
by_group=results.groupby(['source_group','arm'],as_index=False).agg(pages=('page','nunique'),true_print_recovered=('true_print_recovered','sum'),annotation_retained=('annotation_retained','sum'),truth_uncertain_retained=('truth_uncertain_retained','sum'),utility=('utility','sum'),catastrophic_components=('catastrophic_components','sum'))
by_group['false_true_ratio']=by_group.annotation_retained/np.maximum(by_group.true_print_recovered,1);by_group.to_csv(TABLES/'rescored_results_by_source_group.csv',index=False)
manifest_path=ROOT/'development_score_manifest.csv'
if manifest_path.exists():
    m=pd.read_csv(manifest_path,dtype=str).fillna('');col='acquisition_notes' if 'acquisition_notes' in m.columns else None
    if col:
        by_ann=results.merge(m[['page',col]].drop_duplicates('page').rename(columns={col:'annotator'}),on='page',how='left').groupby(['annotator','arm'],as_index=False).agg(pages=('page','nunique'),true_print_recovered=('true_print_recovered','sum'),annotation_retained=('annotation_retained','sum'),utility=('utility','sum'),catastrophic_components=('catastrophic_components','sum'))
        by_ann['false_true_ratio']=by_ann.annotation_retained/np.maximum(by_ann.true_print_recovered,1);by_ann.to_csv(TABLES/'rescored_results_by_annotator.csv',index=False);print(by_ann.to_string(index=False))
print(by_group.to_string(index=False))

## 10. Corrected charts and focused review queue

In [ ]:
sns.set_theme(style='whitegrid')
q=summary.set_index('arm').reindex(['C1_RING2','A2_PAPER_VETO']).reset_index();x=np.arange(len(q))
fig,ax=plt.subplots(figsize=(9,6));ax.bar(x-.2,q.true_print_recovered,.4,label='true print recovered');ax.bar(x+.2,q.annotation_retained,.4,label='annotation retained');ax.set_xticks(x,['C1 Ring2','A2 paper veto']);ax.set_title('Exact Notebook 75 dark-collar rescoring');ax.legend();plt.tight_layout();plt.savefig(VISUAL/'corrected_arm_benefit_cost.png',dpi=180);plt.close(fig)

# Page deltas and review ranking.
piv=results.pivot(index=['page_key','page','source_group'],columns='arm',values=['true_print_recovered','annotation_retained','truth_uncertain_retained','utility']).reset_index();piv.columns=['_'.join(str(x) for x in c if str(x)) for c in piv.columns]
piv['true_print_delta_A2_vs_C1']=piv['true_print_recovered_A2_PAPER_VETO']-piv['true_print_recovered_C1_RING2'];piv['annotation_delta_A2_vs_C1']=piv['annotation_retained_A2_PAPER_VETO']-piv['annotation_retained_C1_RING2'];piv['utility_delta_A2_vs_C1']=piv['utility_A2_PAPER_VETO']-piv['utility_C1_RING2'];piv=piv.merge(removed,on=['page_key','page','source_group'],validate='one_to_one');piv=piv.sort_values(['annotation_removed','true_print_removed'],ascending=[False,False]);piv.to_csv(TABLES/'corrected_visual_review_queue.csv',index=False)

plt.figure(figsize=(9,7));sns.scatterplot(data=piv,x='true_print_removed',y='annotation_removed',size='c1_dark_selected_removed',hue='source_group',sizes=(30,280),legend=False);plt.axvline(0,color='black',linewidth=1);plt.title('What the locked paper veto removes under the exact C1 population');plt.tight_layout();plt.savefig(VISUAL/'paper_veto_removal_tradeoff.png',dpi=180);plt.close()

# Whole-page queue uses locked images, not regenerated arms.
rows=piv.head(min(12,len(piv)));fig,axes=plt.subplots(len(rows),4,figsize=(18,max(5,5.3*len(rows))),squeeze=False)
for i,(_,r) in enumerate(rows.iterrows()):
    d=NB77/'pages'/r.page_key;items=[('C0_INCUMBENT.png','C0'),('C1_RING2.png','C1'),('A2_PAPER_VETO.png','A2'),('observable_context_overlay.png','locked context')]
    for j,(fn,title) in enumerate(items):
        im=cv2.imread(str(d/fn),cv2.IMREAD_UNCHANGED)
        if im is not None and im.ndim==3:im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
        axes[i,j].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key} {title}\nannotation removed={int(r.annotation_removed)} true lost={int(r.true_print_removed)}');axes[i,j].axis('off')
plt.tight_layout();plt.savefig(REVIEW/'corrected_paper_veto_review_queue.png',dpi=130,bbox_inches='tight');plt.close(fig)

## 11. Verdict record

In [ ]:
if comparison['improves_without_any_genuine_print_loss']:
    interpretation='Under exact reproduction of the Notebook 75 dark-collar truth population, the locked A2 observable-paper veto improves C1 by removing retained annotation while preserving every genuine-print pixel recovered by C1 on every page. A2 is the stronger development candidate, subject to visual review and fresh independent/shadow evidence before operational adoption.'
elif comparison['true_print_delta']<0 and comparison['annotation_delta']<0:
    interpretation='A2 removes annotation but also sacrifices genuine C1 print. The paper veto is a trade-off rather than a strict improvement; inspect page/component concentration before deciding between C1 and A2.'
elif comparison['annotation_delta']>=0:
    interpretation='A2 does not reduce retained annotation under the exact Notebook 75 population. Retain C1 as the stronger candidate and treat the current paper-veto definition as unsuccessful.'
else:
    interpretation='A2 and C1 are effectively tied under the exact population. Do not add paper-veto complexity without a demonstrated practical gain.'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook77a-exact-dark-collar-rescore','source_notebook77':str(NB77),'source_notebook77_lock_id':lock77.get('lock_id'),'source_notebook75':str(NB75),'source_notebook75a':str(NB75A),'c1_exact_reproduction_passed':True,'comparison':comparison,'interpretation':interpretation,'observable_policy_changed':False,'operator_changed':False,'promotion_authority':False}
atomic_json(OUT/'notebook77a_verdict.json',record)
lines=['NOTEBOOK 77A — EXACT DARK-COLLAR RESCORING','',f'Notebook 77: {NB77}',f'Notebook 77 lock: {lock77.get("lock_id")}',f'Notebook 75: {NB75}','',f'C1 exact pagewise reproduction: {exact_pagewise}',f'C1 exact pooled reproduction: {pooled_exact}','',summary.to_string(index=False),'','A2 VERSUS C1',json.dumps(comparison,indent=2),'','INTERPRETATION',interpretation,'','Development scoring repair only. Locked Notebook 77 decisions were not rebuilt or changed.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))

## 12. Standard final upload package

In [ ]:
artefacts=[OUT/'executive_summary.txt',OUT/'notebook77a_verdict.json',AUDIT/'notebook77_lock_replay.csv',AUDIT/'notebook75_c1_reference.json',AUDIT/'c1_exact_reproduction_by_page.csv',AUDIT/'c1_reproduction_gate.json',TABLES/'rescored_results_by_page.csv',TABLES/'rescored_pooled_arm_summary.csv',TABLES/'A2_vs_C1_exact_comparison.json',TABLES/'paper_veto_removed_dark_population.csv',TABLES/'rescored_results_by_source_group.csv',TABLES/'corrected_visual_review_queue.csv',VISUAL/'corrected_arm_benefit_cost.png',VISUAL/'paper_veto_removal_tradeoff.png',REVIEW/'corrected_paper_veto_review_queue.png']
if (TABLES/'rescored_results_by_annotator.csv').exists():artefacts.append(TABLES/'rescored_results_by_annotator.csv')
for p in artefacts:
    require(p.exists(),f'Missing upload artefact: {p}');shutil.copy2(p,UPLOAD/p.name)
(UPLOAD/'README.txt').write_text('Notebook 77A rescored the already locked Notebook 77 A2 paper-veto decisions using the exact Notebook 75 dark-collar population. Interpretation is permitted only after exact pagewise C1 reproduction. Start with executive_summary.txt, c1_reproduction_gate.json, rescored_pooled_arm_summary.csv, A2_vs_C1_exact_comparison.json and corrected_arm_benefit_cost.png.\n')
manifest=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False);(UPLOAD/'upload_file_list.txt').write_text('\n'.join(sorted(p.name for p in UPLOAD.iterdir() if p.is_file())))
zip_path=UPLOAD/'notebook77a_exact_dark_collar_rescore_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path:z.write(p,p.name)
print('Final bundle:',zip_path)

## Reading Notebook 77A

- If the C1 reproduction gate fails, stop. The scorer has not reproduced Notebook 75 and no A2 conclusion is valid.
- If C1 reproduces exactly and A2 has `true_print_delta = 0`, `annotation_delta < 0`, and zero pages with true-print loss, A2 is a strict development improvement over C1 on this cohort.
- If A2 removes both annotation and true print, inspect whether the sacrificed print is visually material before choosing A2.
- This is still development reuse of the same 51 exposed pages. A strict A2 improvement supports choosing A2 for the next shadow/validation stage, not automatic production promotion.
- The paper-veto inference rule remains observable-only. Clean truth appears only in this 77A scoring repair.